# Tugas UAS - II4013 Data Analytics
## Kelompok 1

| NIM | Nama |
|-----|------|
| 18223061 | Naura Ayurachmani |
| 18223069 | Catherine Alicia N. |
| 18223081 | Aliya Harta A. |
| 18223095 | Noeriza Aqila W. |

## 1. Import Library

In [ ]:
import re

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

## 2. Load Dataset

Dataset bersumber dari tiga file:
- **SIPSN**: data capaian pengelolaan sampah per kabupaten/kota
- **TPA**: data fasilitas Tempat Pemrosesan Akhir sampah
- **Peta**: geometri batas wilayah Indonesia (GeoJSON)

In [ ]:
URL_SIPSN = 'https://drive.google.com/uc?export=download&id=19o6WuM83duQ8EH7BcfunllkAzJXOIbaS'
URL_TPA   = 'https://drive.google.com/uc?export=download&id=1Fndlht7ZHUQMINWfIx1Nma2MA3dSDouW'
URL_MAP   = 'https://drive.google.com/uc?export=download&id=1FwbUxWe7AyzvPQiPcWC--uUsVUsYDFv-'

df_sipsn = pd.read_excel(URL_SIPSN, skiprows=1)
df_tpa   = pd.read_excel(URL_TPA, skiprows=1)
df_map   = gpd.read_file(URL_MAP)

print(f'df_sipsn : {df_sipsn.shape}')
print(f'df_tpa   : {df_tpa.shape}')
print(f'df_map   : {df_map.shape}')

Capaian SIPSN Dataframe
   Tahun Provinsi      Kabupaten/Kota  Timbulan Sampah Tahunan (ton/tahun)(A)  \
0   2025     Aceh   Kab. Aceh Selatan                                35331.12   
1   2025     Aceh  Kab. Aceh Tenggara                                42994.99   
2   2025     Aceh     Kab. Aceh Timur                                67363.09   
3   2025     Aceh    Kab. Aceh Tengah                                41601.58   
4   2025     Aceh     Kab. Aceh Barat                                30802.93   

   Pengurangan Sampah Tahunan (ton/tahun)(B)  %Pengurangan Sampah(B/A)  \
0                                       0.00                       0.0   
1                                       0.00                       0.0   
2                                       1.88                       0.0   
3                                       0.00                       0.0   
4                                       0.61                       0.0   

   Penanganan Sampah Tahunan (ton/tahun)(C) 

### Struktur Dataset

In [ ]:
df_sipsn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 14 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Tahun                                      279 non-null    int64  
 1   Provinsi                                   279 non-null    object 
 2   Kabupaten/Kota                             279 non-null    object 
 3   Timbulan Sampah Tahunan (ton/tahun)(A)     279 non-null    float64
 4   Pengurangan Sampah Tahunan (ton/tahun)(B)  279 non-null    float64
 5   %Pengurangan Sampah(B/A)                   279 non-null    float64
 6   Penanganan Sampah Tahunan (ton/tahun)(C)   279 non-null    float64
 7   %Penanganan Sampah(C/A)                    279 non-null    float64
 8   Sampah Terkelola Tahunan (ton/tahun)(B+C)  279 non-null    float64
 9   %Sampah Terkelola(B+C)/A                   279 non-null    float64
 10  Daur ulang Sampah Tahunan 

In [ ]:
df_tpa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309 entries, 0 to 308
Data columns (total 32 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Unnamed: 0                              0 non-null      float64
 1   Tahun                                   309 non-null    int64  
 2   P                                       309 non-null    int64  
 3   Provinsi                                309 non-null    object 
 4   Kabupaten/Kota                          309 non-null    object 
 5   Nama Fasilitas                          309 non-null    object 
 6   Jenis                                   303 non-null    object 
 7   Status                                  309 non-null    object 
 8   Sampahmasuk (ton/thn)                   309 non-null    float64
 9   Sampahmasuk Landfill (ton/thn)          309 non-null    float64
 10  Sampah Organikterolah (ton/thn)         309 non-null    float6

In [ ]:
df_map.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   GID_2      502 non-null    object  
 1   GID_0      502 non-null    object  
 2   COUNTRY    502 non-null    object  
 3   GID_1      502 non-null    object  
 4   NAME_1     502 non-null    object  
 5   NL_NAME_1  502 non-null    object  
 6   NAME_2     502 non-null    object  
 7   VARNAME_2  502 non-null    object  
 8   NL_NAME_2  502 non-null    object  
 9   TYPE_2     502 non-null    object  
 10  ENGTYPE_2  502 non-null    object  
 11  CC_2       502 non-null    object  
 12  HASC_2     502 non-null    object  
 13  geometry   502 non-null    geometry
dtypes: geometry(1), object(13)
memory usage: 55.0+ KB


## 3. Data Cleaning

Tahapan pembersihan data meliputi:
- Penggabungan (merge) dataset SIPSN dan TPA
- Pengecekan missing values, duplikasi, tipe data, dan outlier
- Standarisasi format kolom kategorikal dan tanggal

### 3.1 Merge Dataset

In [ ]:
df_merged = pd.merge(
    df_sipsn,
    df_tpa,
    on=['Provinsi', 'Kabupaten/Kota'],
    how='outer',
    suffixes=('_sipsn', '_tpa'),
)

print(f'df_sipsn shape : {df_sipsn.shape}')
print(f'df_tpa shape   : {df_tpa.shape}')
print(f'df_merged shape: {df_merged.shape}')
df_merged.head()

df_sipsn shape : (279, 14)
df_tpa shape   : (309, 32)
df_merged shape: (370, 44)
   Tahun_sipsn Provinsi        Kabupaten/Kota  \
0       2025.0     Aceh       Kab. Aceh Barat   
1       2025.0     Aceh  Kab. Aceh Barat Daya   
2       2025.0     Aceh       Kab. Aceh Besar   
3       2025.0     Aceh        Kab. Aceh Jaya   
4       2025.0     Aceh        Kab. Aceh Jaya   

   Timbulan Sampah Tahunan (ton/tahun)(A)  \
0                                30802.93   
1                                29444.55   
2                                55435.47   
3                                18506.78   
4                                18506.78   

   Pengurangan Sampah Tahunan (ton/tahun)(B)  %Pengurangan Sampah(B/A)  \
0                                       0.61                      0.00   
1                                     228.13                      0.77   
2                                       4.62                      0.01   
3                                       0.00             

In [ ]:
df_merged.to_csv('Dataset/merged_waste_management.csv', index=False)
df = pd.read_csv('Dataset/merged_waste_management.csv')

### 3.2 Pemeriksaan Awal

In [ ]:
print(df.dtypes)
print(f'\nTotal kolom: {len(df.columns)}')

Tahun_sipsn                                  float64
Provinsi                                      object
Kabupaten/Kota                                object
Timbulan Sampah Tahunan (ton/tahun)(A)       float64
Pengurangan Sampah Tahunan (ton/tahun)(B)    float64
%Pengurangan Sampah(B/A)                     float64
Penanganan Sampah Tahunan (ton/tahun)(C)     float64
%Penanganan Sampah(C/A)                      float64
Sampah Terkelola Tahunan (ton/tahun)(B+C)    float64
%Sampah Terkelola(B+C)/A                     float64
Daur ulang Sampah Tahunan (ton/tahun)(D)     float64
Bahan baku Sampah Tahunan (ton/tahun)(E)     float64
Recycling Rate(D+E)/A                        float64
P1/P2                                         object
Unnamed: 0                                   float64
Tahun_tpa                                    float64
P                                            float64
Nama Fasilitas                                object
Jenis                                         

#### Missing Values

In [ ]:
missing     = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct,
}).sort_values('Persentase (%)', ascending=False)

print('Missing Values')
print(missing_df[missing_df['Jumlah Missing'] > 0])
print(f'\nTotal baris          : {len(df)}')
print(f'Kolom tanpa missing  : {(missing == 0).sum()}')
print(f'Kolom dengan missing : {(missing > 0).sum()}')

=== MISSING VALUES ===
                                           Jumlah Missing  Persentase (%)
Unnamed: 0                                            370          100.00
Kelurahan                                             184           49.73
Kecamatan                                             184           49.73
Pengelola                                              82           22.16
Jenis                                                  67           18.11
Sampah Organikterolah (ton/thn)                        61           16.49
Tgl Awal Operasi                                       61           16.49
Energi(MW)                                             61           16.49
Alamat                                                 61           16.49
Luas (hektar)                                          61           16.49
Sistem Operasional                                     61           16.49
Tgl. Akhir Operasi                                     61           16.49
Sampah An-Organ

#### Duplikasi

In [ ]:
print(f'Duplikat (semua kolom)                              : {df.duplicated().sum()}')
print(f'Duplikat (Provinsi + Kabupaten/Kota + Nama Fasilitas): '
      f"{df.duplicated(subset=['Provinsi', 'Kabupaten/Kota', 'Nama Fasilitas']).sum()}")

dupes = df[df.duplicated(keep=False)]
if not dupes.empty:
    print(f'\nBaris duplikat ditemukan ({len(dupes)} baris):')
    print(dupes[['Provinsi', 'Kabupaten/Kota', 'Nama Fasilitas']].head(10))
else:
    print('\nTidak ada baris duplikat.')

Jumlah baris duplikat (semua kolom): 0
Jumlah baris duplikat (Provinsi + Kabupaten/Kota + Nama Fasilitas): 0

 Tidak ada baris duplikat.


#### Distribusi Wilayah

In [ ]:
print(f'Unique Provinsi      : {df["Provinsi"].nunique()}')
print(sorted(df['Provinsi'].dropna().unique()))

print(f'\nUnique Kabupaten/Kota: {df["Kabupaten/Kota"].nunique()}')
print(sorted(df['Kabupaten/Kota'].dropna().unique())[:20])

Unique Provinsi: 37
['Aceh', 'Bali', 'Banten', 'Bengkulu', 'DKI Jakarta', 'Daerah Istimewa Yogyakarta', 'Gorontalo', 'Jambi', 'Jawa Barat', 'Jawa Tengah', 'Jawa Timur', 'Kalimantan Barat', 'Kalimantan Selatan', 'Kalimantan Tengah', 'Kalimantan Timur', 'Kalimantan Utara', 'Kepulauan Bangka Belitung', 'Kepulauan Riau', 'Lampung', 'Maluku', 'Maluku Utara', 'Nusa Tenggara Barat', 'Nusa Tenggara Timur', 'Papua', 'Papua Barat', 'Papua Barat Daya', 'Papua Selatan', 'Papua Tengah', 'Riau', 'Sulawesi Barat', 'Sulawesi Selatan', 'Sulawesi Tengah', 'Sulawesi Tenggara', 'Sulawesi Utara', 'Sumatera Barat', 'Sumatera Selatan', 'Sumatera Utara']

Unique Kabupaten/Kota: 315
['Kab. Aceh Barat', 'Kab. Aceh Barat Daya', 'Kab. Aceh Besar', 'Kab. Aceh Jaya', 'Kab. Aceh Selatan', 'Kab. Aceh Singkil', 'Kab. Aceh Tamiang', 'Kab. Aceh Tengah', 'Kab. Aceh Tenggara', 'Kab. Aceh Timur', 'Kab. Aceh Utara', 'Kab. Adm. Kep. Seribu', 'Kab. Agam', 'Kab. Alor', 'Kab. Asmat', 'Kab. Badung', 'Kab. Balangan', 'Kab. Bandun

#### Format Tanggal

In [ ]:
print('Tgl Awal Operasi (sample):')
print(df['Tgl Awal Operasi'].dropna().unique()[:20])

print('\nTgl. Akhir Operasi (sample):')
print(df['Tgl. Akhir Operasi'].dropna().unique()[:20])

print('\nCatatan: Format tanggal tidak konsisten (tahun saja, dd/mm/yyyy, teks, dll).')

=== FORMAT TANGGAL ===

--- Tgl Awal Operasi (sample values) ---
['2022' '2013' '2015' '2006' '-' '2010' '26 Februari 2016' 'TAHUN 2013'
 '2016' '2011' '1 Maret 2008' '2020' '1 Januari 2019' '01/01/2014'
 '17-10-2012' 'JANUARI 2012' '15 November 2017' '13 April' 'Januari 2010'
 '2008']

--- Tgl. Akhir Operasi (sample values) ---
['-' '2036' '30 Juni 2022' '2024' '2030' '2026' '2032' '2021'
 'Desember 2030' '2016' '2027' '2035' '2033' '2023' '01-01-2038'
 '1 Januari 2025' '2022' '2025' '2020' '5']

  Format tanggal kemungkinan tidak konsisten (campuran tahun saja, dd/mm/yyyy, teks, dll)


#### Kolom Kategorikal

In [ ]:
KATEGORI_COLS = [
    'Jenis', 'Status', 'Sistem Operasional', 'Pencatatan',
    'Jembatan Timbang', 'Penutupan Sampah Zona Aktif',
    'Ada Drainase', 'P1/P2', 'Pengelola',
]

for col in KATEGORI_COLS:
    if col in df.columns:
        print(f'--- {col} ---')
        print(df[col].value_counts(dropna=False))
        print()

=== KOLOM KATEGORIKAL ===

--- Jenis ---
Jenis
TPA Pemda (Non Regional)    280
NaN                          67
TPA Regional                 23
Name: count, dtype: int64

--- Status ---
Status
A      309
NaN     61
Name: count, dtype: int64

--- Sistem Operasional ---
Sistem Operasional
Open Dumping         166
Control Landfill     102
NaN                   61
-                     30
Sanitary Landfill     11
Name: count, dtype: int64

--- Pencatatan ---
Pencatatan
Manual Book                                                     165
Kombinasi timbangan dengan Komputer (excel / tidak otomatis)     89
NaN                                                              61
Otomatis Komputerisasi (Aplikasi Khusus TPA)                     42
-                                                                13
Name: count, dtype: int64

--- Jembatan Timbang ---
Jembatan Timbang
Tidak Ada                        140
Ada dan berfungsi dengan baik    125
NaN                               61
Ada tapi se

#### Statistik Deskriptif dan Outlier (Metode IQR)

In [ ]:
numerik_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print('Statistik Deskriptif (Numerik)')
print(df[numerik_cols].describe().round(2))

print('\nDeteksi Outlier (IQR Method)')
for col in numerik_cols:
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    if n_out > 0:
        print(f'  {col}: {n_out} outlier (range normal: {lower:.2f} - {upper:.2f})')

=== STATISTIK DESKRIPTIF (NUMERIK) ===
       Tahun_sipsn  Timbulan Sampah Tahunan (ton/tahun)(A)  \
count        329.0                                  329.00   
mean        2025.0                               113091.78   
std            0.0                               145678.89   
min         2025.0                                 4425.99   
25%         2025.0                                31626.81   
50%         2025.0                                59240.96   
75%         2025.0                               142935.17   
max         2025.0                              1037020.30   

       Pengurangan Sampah Tahunan (ton/tahun)(B)  %Pengurangan Sampah(B/A)  \
count                                     329.00                    329.00   
mean                                     1988.68                      1.61   
std                                      9812.67                      5.43   
min                                         0.00                      0.00   
25%         

### 3.3 Pembersihan Data

#### Drop Kolom Tidak Relevan

In [ ]:
print(f"'Unnamed: 0' non-null: {df['Unnamed: 0'].notna().sum()} -> di-drop")
print(f"'P'          non-null: {df['P'].notna().sum()} -> di-drop")
print("'Tahun_sipsn' dan 'Tahun_tpa' akan digabung menjadi kolom 'Tahun'.")

- 'Unnamed: 0': 0 non-null values → kemungkinan bisa di-drop
- 'P': 309 non-null values → cek apakah perlu
- 'Tahun_sipsn' vs 'Tahun_tpa': apakah perlu digabung jadi 1 kolom 'Tahun'?


In [ ]:
df_clean = df.drop(columns=['Unnamed: 0', 'P'])
print(f"Kolom 'Unnamed: 0' dan 'P' berhasil di-drop.")

 Drop kolom 'Unnamed: 0' dan 'P'


#### Konsolidasi Kolom Tahun

In [ ]:
df_clean['Tahun'] = df_clean['Tahun_sipsn'].fillna(df_clean['Tahun_tpa']).astype(int)
df_clean = df_clean.drop(columns=['Tahun_sipsn', 'Tahun_tpa'])

# Pindahkan kolom 'Tahun' ke posisi pertama
cols     = ['Tahun'] + [c for c in df_clean.columns if c != 'Tahun']
df_clean = df_clean[cols]

print("Kolom 'Tahun_sipsn' dan 'Tahun_tpa' digabung menjadi 'Tahun'.")
print(df_clean['Tahun'].value_counts())

 Gabung Tahun_sipsn & Tahun_tpa → 'Tahun'
Tahun
2025    370
Name: count, dtype: int64


#### Normalisasi Nilai Sentinel

In [ ]:
df_clean = df_clean.replace('-', np.nan)

missing_after = df_clean.isnull().sum()
print("Missing values setelah mengganti '-' dengan NaN:")
print(missing_after[missing_after > 0].sort_values(ascending=False))

 Ganti semua '-' dengan NaN

Missing values setelah replace '-':
Jumlah KK yang memanfaatkan gas Metana       351
Tgl. Akhir Operasi                           287
Jml Uji Lindi                                214
Kelurahan                                    185
Kecamatan                                    184
Pemanfaatan Gas Metana                       180
Tgl Awal Operasi                             179
Jml Sumur Pantau                             172
IPL                                          100
Luas Landfill Aktif (m2)                      97
Sistem Operasional                            91
Pengelola                                     82
Pencatatan                                    74
Jembatan Timbang                              71
Penutupan Sampah Zona Aktif                   69
Jenis                                         67
Luas (hektar)                                 62
Ada Drainase                                  61
Alamat                                        61
Ener

#### Konversi Tipe Data Numerik

In [ ]:
KOLOM_NUMERIK = [
    'Luas Landfill Aktif (m2)', 'Jml Sumur Pantau', 'Jml Uji Lindi',
    'Luas (hektar)', 'Energi(MW)', 'Sampahmasuk (ton/thn)',
    'Sampahmasuk Landfill (ton/thn)', 'Sampah Organikterolah (ton/thn)',
    'Sampah An-Organikterolah (ton/thn)', 'RecoveryPemulung (ton/thn)',
    'Jumlah KK yang memanfaatkan gas Metana',
]

for col in KOLOM_NUMERIK:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print('Tipe data setelah konversi:')
print(df_clean[KOLOM_NUMERIK].dtypes)

 Konversi kolom numerik ke tipe numeric

Tipe data setelah konversi:
Luas Landfill Aktif (m2)                  float64
Jml Sumur Pantau                          float64
Jml Uji Lindi                             float64
Luas (hektar)                             float64
Energi(MW)                                float64
Sampahmasuk (ton/thn)                     float64
Sampahmasuk Landfill (ton/thn)            float64
Sampah Organikterolah (ton/thn)           float64
Sampah An-Organikterolah (ton/thn)        float64
RecoveryPemulung (ton/thn)                float64
Jumlah KK yang memanfaatkan gas Metana    float64
dtype: object


#### Ekstraksi Tahun dari Kolom Tanggal

In [ ]:
def extract_year(val: object) -> float:
    """Ekstrak tahun (4 digit) dari string tanggal dengan format tidak konsisten."""
    if pd.isna(val):
        return np.nan
    match = re.search(r'(19|20)\d{2}', str(val).strip())
    return int(match.group()) if match else np.nan


df_clean['Tahun Awal Operasi']  = df_clean['Tgl Awal Operasi'].apply(extract_year)
df_clean['Tahun Akhir Operasi'] = df_clean['Tgl. Akhir Operasi'].apply(extract_year)
df_clean = df_clean.drop(columns=['Tgl Awal Operasi', 'Tgl. Akhir Operasi'])

print(f'Tahun Awal Operasi  - range: '
      f"{df_clean['Tahun Awal Operasi'].min():.0f} - {df_clean['Tahun Awal Operasi'].max():.0f}")
print(f'Tahun Akhir Operasi - range: '
      f"{df_clean['Tahun Akhir Operasi'].min():.0f} - {df_clean['Tahun Akhir Operasi'].max():.0f}")

df_clean[['Nama Fasilitas', 'Tahun Awal Operasi', 'Tahun Akhir Operasi']].dropna().head(10)

✅ Ekstrak tahun dari tanggal operasi

Tahun Awal Operasi - range: 1972 - 2024
Tahun Akhir Operasi - range: 2016 - 2046

Contoh:
         Nama Fasilitas  Tahun Awal Operasi  Tahun Akhir Operasi
16        TPA BABAH DUA              2016.0               2036.0
17  TPA Cot Padang Lila              2012.0               2022.0
24       TPA LHOK BATEE              2000.0               2024.0
37             Cilowong              1995.0               2026.0
41                  TPA              2012.0               2032.0
43      TPA Air Sebakul              1992.0               2032.0
66     TPA Gunungsantri              1986.0               2021.0
75           TPA Galuga              2010.0               2030.0
77     UPT TPA Cipayung              1992.0               2016.0
79           TPA WINONG              1990.0               2027.0


#### Standarisasi Kolom Kategorikal

In [ ]:
def simplify_jembatan(val: object) -> object:
    """Sederhanakan nilai kolom 'Jembatan Timbang' menjadi 3 kategori."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if 'berfungsi' in val or 'ada dan' in val:
        return 'Ada & Berfungsi'
    if 'rusak' in val:
        return 'Ada tapi Rusak'
    if 'tidak' in val:
        return 'Tidak Ada'
    return val.title()


def simplify_penutupan(val: object) -> object:
    """Sederhanakan nilai kolom 'Penutupan Sampah Zona Aktif'."""
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    mapping = {
        'setiap hari':              'Setiap Hari',
        'tiga hari':                '2-3x Seminggu',
        'seminggu dua':             '2-3x Seminggu',
        'tujuh hari':               'Seminggu Sekali',
        'seminggu sekali':          'Seminggu Sekali',
        'dua minggu':               'Dua Minggu Sekali',
        'sebulan':                  'Sebulan Sekali',
        'tiga atau empat bulan':    '3-4 Bulan Sekali',
        'setahun':                  'Setahun Sekali',
        'tidak ditutup':            'Tidak Ditutup',
    }
    for key, label in mapping.items():
        if key in val:
            return label
    return val.title()


if 'Ada Drainase' in df_clean.columns:
    df_clean['Ada Drainase'] = df_clean['Ada Drainase'].str.strip().str.upper()

df_clean['Jembatan Timbang']            = df_clean['Jembatan Timbang'].apply(simplify_jembatan)
df_clean['Penutupan Sampah Zona Aktif'] = df_clean['Penutupan Sampah Zona Aktif'].apply(simplify_penutupan)

print('--- Jembatan Timbang ---')
print(df_clean['Jembatan Timbang'].value_counts(dropna=False))

print('\n--- Penutupan Sampah Zona Aktif ---')
print(df_clean['Penutupan Sampah Zona Aktif'].value_counts(dropna=False))

print('\n--- Ada Drainase ---')
print(df_clean['Ada Drainase'].value_counts(dropna=False))

 Standarisasi kolom kategorikal

--- Jembatan Timbang ---
Jembatan Timbang
Tidak Ada          140
Ada & Berfungsi    125
NaN                 71
Ada tapi Rusak      34
Name: count, dtype: int64

--- Penutupan Sampah Zona Aktif ---
Penutupan Sampah Zona Aktif
NaN                  69
Seminggu Sekali      67
Tidak Ditutup        54
Sebulan Sekali       42
2-3x Seminggu        40
Dua Minggu Sekali    38
Setahun Sekali       26
3-4 Bulan Sekali     23
Setiap Hari          11
Name: count, dtype: int64

--- Ada Drainase ---
Ada Drainase
ADA          228
TIDAK ADA     81
NaN           61
Name: count, dtype: int64


#### Penandaan Ketersediaan Data SIPSN dan TPA

In [ ]:
KOLOM_SIPSN = [
    'Timbulan Sampah Tahunan (ton/tahun)(A)',
    'Pengurangan Sampah Tahunan (ton/tahun)(B)',
    '%Pengurangan Sampah(B/A)',
    'Penanganan Sampah Tahunan (ton/tahun)(C)',
    '%Penanganan Sampah(C/A)',
    'Sampah Terkelola Tahunan (ton/tahun)(B+C)',
    '%Sampah Terkelola(B+C)/A',
    'Daur ulang Sampah Tahunan (ton/tahun)(D)',
    'Bahan baku Sampah Tahunan (ton/tahun)(E)',
    'Recycling Rate(D+E)/A',
]

df_clean['Ada Data SIPSN'] = df_clean[KOLOM_SIPSN[0]].notna()
df_clean['Ada Data TPA']   = df_clean['Nama Fasilitas'].notna()

print(f"Baris dengan data SIPSN : {df_clean['Ada Data SIPSN'].sum()}")
print(f"Baris dengan data TPA   : {df_clean['Ada Data TPA'].sum()}")
print(f"Baris dengan keduanya   : {(df_clean['Ada Data SIPSN'] & df_clean['Ada Data TPA']).sum()}")
print(f"Baris SIPSN saja        : {(df_clean['Ada Data SIPSN'] & ~df_clean['Ada Data TPA']).sum()}")
print(f"Baris TPA saja          : {(~df_clean['Ada Data SIPSN'] & df_clean['Ada Data TPA']).sum()}")

✅ Tandai ketersediaan data SIPSN dan TPA

Baris dengan data SIPSN: 329
Baris dengan data TPA : 309
Baris dengan keduanya : 268
Baris SIPSN saja      : 61
Baris TPA saja        : 41


#### Imputasi Missing Value pada Kolom Kategorikal

In [ ]:
df_clean['Pengelola']          = df_clean['Pengelola'].fillna('Tidak Diketahui')
df_clean['Sistem Operasional'] = df_clean['Sistem Operasional'].fillna('Tidak Diketahui')

print('--- Pengelola ---')
print(df_clean['Pengelola'].value_counts())

print('\n--- Sistem Operasional ---')
print(df_clean['Sistem Operasional'].value_counts())

 Isi missing value kolom Pengelola & Sistem Operasional
Pengelola
Pemda DATI II      256
Tidak Diketahui     82
Pemda DATI I        16
Lainnya             15
Swasta               1
Name: count, dtype: int64

Sistem Operasional
Open Dumping         166
Control Landfill     102
Tidak Diketahui       91
Sanitary Landfill     11
Name: count, dtype: int64


#### Pengecekan dan Penghapusan Duplikat

In [ ]:
n_dupes = df_clean.duplicated().sum()
print(f'Duplikat sempurna (semua kolom): {n_dupes}')

if n_dupes > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'{n_dupes} duplikat sempurna dihapus.')
else:
    print('Tidak ada duplikat sempurna, tidak perlu penghapusan.')

=== CEK DUPLIKASI ===
Duplikat sempurna (semua kolom): 0
 Tidak ada duplikat sempurna — tidak perlu penghapusan


#### Penanganan Outlier

> Outlier **ditandai namun tidak dihapus**. Variasi data sampah antar daerah memang sangat besar; menghapus outlier berarti menghapus data kabupaten/kota besar yang valid.

In [ ]:
EXCLUDE_COLS  = ['Tahun', 'Ada Data SIPSN', 'Ada Data TPA']
numerik_clean = (
    df_clean.select_dtypes(include=[np.number])
    .drop(columns=EXCLUDE_COLS, errors='ignore')
)

outlier_flags = {}
for col in numerik_clean.columns:
    Q1    = df_clean[col].quantile(0.25)
    Q3    = df_clean[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    if n_out > 0:
        outlier_flags[col] = n_out

print('Outlier per kolom (IQR method):')
for col, count in sorted(outlier_flags.items(), key=lambda x: x[1], reverse=True):
    print(f'  {col}: {count} outlier')

=== OUTLIER (ditandai, TIDAK dihapus) ===
Alasan: variasi data sampah antar daerah memang sangat besar,
menghapus outlier = menghapus data kota/kab besar yang valid.

  Sampah An-Organikterolah (ton/thn): 77 outlier
  %Pengurangan Sampah(B/A): 69 outlier
  Daur ulang Sampah Tahunan (ton/tahun)(D): 69 outlier
  Pengurangan Sampah Tahunan (ton/tahun)(B): 64 outlier
  Sampah Organikterolah (ton/thn): 57 outlier
  Luas (hektar): 44 outlier
  RecoveryPemulung (ton/thn): 39 outlier
  Sampahmasuk (ton/thn): 38 outlier
  Sampahmasuk Landfill (ton/thn): 37 outlier
  Sampah Terkelola Tahunan (ton/tahun)(B+C): 33 outlier
  Penanganan Sampah Tahunan (ton/tahun)(C): 31 outlier
  Bahan baku Sampah Tahunan (ton/tahun)(E): 31 outlier
  Timbulan Sampah Tahunan (ton/tahun)(A): 27 outlier
  Luas Landfill Aktif (m2): 20 outlier
  Recycling Rate(D+E)/A: 12 outlier
  Jml Sumur Pantau: 12 outlier
  Jml Uji Lindi: 9 outlier
  Tahun Awal Operasi: 4 outlier
  Jumlah KK yang memanfaatkan gas Metana: 3 outlier
  

### 3.4 Ringkasan Data Cleaning

In [ ]:
print(f'Shape awal  : {df.shape}')
print(f'Shape akhir : {df_clean.shape}')

print('\nKolom di-drop    : Unnamed: 0, P, Tgl Awal Operasi, Tgl. Akhir Operasi, Tahun_sipsn, Tahun_tpa')
print('Kolom ditambah   : Tahun, Tahun Awal Operasi, Tahun Akhir Operasi, Ada Data SIPSN, Ada Data TPA')

missing_final = df_clean.isnull().sum()
print('\nMissing values tersisa:')
print(missing_final[missing_final > 0].sort_values(ascending=False))

print('\nTipe data akhir:')
print(df_clean.dtypes)

df_clean.to_csv('Dataset/cleaned_waste_management.csv', index=False)
print('\nDataset disimpan: Dataset/cleaned_waste_management.csv')

RINGKASAN DATA CLEANING
Shape awal  : (370, 44)
Shape akhir : (370, 43)

Kolom di-drop: Unnamed: 0, P, Tgl Awal Operasi, Tgl. Akhir Operasi, Tahun_sipsn, Tahun_tpa
Kolom ditambah: Tahun, Tahun Awal Operasi, Tahun Akhir Operasi, Ada Data SIPSN, Ada Data TPA

Missing values tersisa:
Jumlah KK yang memanfaatkan gas Metana       351
Tahun Akhir Operasi                          299
Jml Uji Lindi                                214
Kelurahan                                    185
Kecamatan                                    184
Tahun Awal Operasi                           180
Pemanfaatan Gas Metana                       180
Jml Sumur Pantau                             172
IPL                                          100
Luas Landfill Aktif (m2)                      97
Pencatatan                                    74
Jembatan Timbang                              71
Penutupan Sampah Zona Aktif                   69
Jenis                                         67
Luas (hektar)                   

## 4. Feature Selection

Tahap ini mengidentifikasi dan menghapus fitur yang tidak memberi nilai tambah pada model.

| Kriteria Drop | Alasan |
|---|---|
| **Missing > 50%** | Imputasi akan menambah bias yang signifikan |
| **Near-zero variance** | Hampir semua nilai sama — tidak diskriminatif untuk model |
| **Kolom derivatif** | Dihitung dari kolom lain — menyebabkan multikolinearitas |
| **Identifier / teks** | Penanda unik, bukan fitur prediktif |

In [ ]:
df_clean = pd.read_csv('Dataset/cleaned_waste_management.csv')

EXCLUDE_COLS = ['Tahun', 'Ada Data SIPSN', 'Ada Data TPA']
numerik = (
    df_clean.select_dtypes(include=[np.number])
    .drop(columns=EXCLUDE_COLS, errors='ignore')
)

print(f'Shape sebelum feature selection: {df_clean.shape}')
print(f'Jumlah kolom numerik           : {len(numerik.columns)}')

### 4.1 Matriks Korelasi (Sebelum Feature Selection)

In [ ]:
corr = numerik.corr()

fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, annot_kws={'size': 7},
    ax=ax,
)
ax.set_title('Matriks Korelasi - Sebelum Feature Selection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Identifikasi Kolom yang Akan Di-drop

In [ ]:
# --- Missing > 50% ---
missing_ratio     = (df_clean.isnull().sum() / len(df_clean) * 100).round(1)
KOLOM_HIGH_MISSING = missing_ratio[missing_ratio > 50].index.tolist()

# --- Near-zero variance: Tahun semua bernilai 2025 ---
KOLOM_LOW_VARIANCE = []
for col in df_clean.select_dtypes(include=[np.number]).columns:
    data = df_clean[col].dropna()
    if data.empty:
        continue
    if data.value_counts().iloc[0] / len(data) > 0.95:
        KOLOM_LOW_VARIANCE.append(col)

# --- Kolom derivatif yang masih tersisa ---
KOLOM_DERIVATIF = [
    'Sampah Terkelola Tahunan (ton/tahun)(B+C)',
    '%Sampah Terkelola(B+C)/A',
    '%Pengurangan Sampah(B/A)',
    '%Penanganan Sampah(C/A)',
    'Recycling Rate(D+E)/A',
]
KOLOM_DERIVATIF = [c for c in KOLOM_DERIVATIF if c in df_clean.columns]

# --- Identifier / teks ---
KOLOM_IDENTIFIER = ['Nama Fasilitas', 'Alamat', 'Kelurahan', 'Kecamatan',
                    'IPL', 'Pencatatan', 'Pemanfaatan Gas Metana']
KOLOM_IDENTIFIER = [c for c in KOLOM_IDENTIFIER if c in df_clean.columns]

print('Kolom high missing (>50%) :', KOLOM_HIGH_MISSING)
print('Kolom near-zero variance  :', KOLOM_LOW_VARIANCE)
print('Kolom derivatif           :', KOLOM_DERIVATIF)
print('Kolom identifier/teks     :', KOLOM_IDENTIFIER)

### 4.3 Eksekusi Feature Selection

In [ ]:
ALL_DROP = list(dict.fromkeys(
    KOLOM_HIGH_MISSING + KOLOM_LOW_VARIANCE + KOLOM_DERIVATIF + KOLOM_IDENTIFIER
))
ALL_DROP = [c for c in ALL_DROP if c in df_clean.columns]

df_selected = df_clean.drop(columns=ALL_DROP)

print(f'Shape sebelum : {df_clean.shape}')
print(f'Shape sesudah  : {df_selected.shape}')
print(f'Kolom di-drop  : {ALL_DROP}')
print(f'\nKolom tersisa ({len(df_selected.columns)}):')
print(df_selected.columns.tolist())

### 4.4 Matriks Korelasi (Setelah Feature Selection)

In [ ]:
numerik_selected = (
    df_selected.select_dtypes(include=[np.number])
    .drop(columns=EXCLUDE_COLS, errors='ignore')
)

corr_sel = numerik_selected.corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr_sel, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5, square=True,
    ax=ax,
)
ax.set_title('Matriks Korelasi - Setelah Feature Selection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

Fitur baru dibuat berdasarkan domain knowledge pengelolaan sampah. Kolom raw yang digantikan
oleh fitur baru akan di-drop di akhir section ini agar tidak ada redundansi.

| Kelompok | Fitur Baru | Penjelasan |
|---|---|---|
| **Efisiensi pengelolaan** | `rasio_pengelolaan`, `rasio_pengurangan`, `rasio_penanganan`, `recycling_rate` | Normalisasi terhadap timbulan sehingga dapat dibandingkan antar daerah |
| **Efisiensi TPA** | `utilisasi_tpa`, `efisiensi_landfill`, `rasio_recovery_pemulung`, `rasio_organik_terolah` | Mengukur seberapa optimal fasilitas TPA dioperasikan |
| **Usia & kapasitas TPA** | `usia_operasi_tpa`, `sampah_per_hektar`, `proporsi_landfill_aktif` | Proksi kondisi fisik dan kepadatan operasional TPA |
| **Skor infrastruktur** | `skor_penutupan`, `skor_jembatan`, `skor_drainase`, `skor_infrastruktur` | Encoding ordinal — mengubah kategori operasional jadi skala numerik bermakna |
| **One-hot encoding** | `Jenis_*`, `Pengelola_*`, `Sistem Operasional_*` | Encoding nominal untuk kolom kategorikal |

> **Catatan**: Semua rasio di-clip ke `[0, 1]` untuk menangani noise akibat data kotor.

In [ ]:
df_fe = df_selected.copy()

# Referensi nama kolom SIPSN
A = 'Timbulan Sampah Tahunan (ton/tahun)(A)'
B = 'Pengurangan Sampah Tahunan (ton/tahun)(B)'
C = 'Penanganan Sampah Tahunan (ton/tahun)(C)'
D = 'Daur ulang Sampah Tahunan (ton/tahun)(D)'
E = 'Bahan baku Sampah Tahunan (ton/tahun)(E)'

### 5.1 Fitur Efisiensi Pengelolaan Sampah (berbasis SIPSN)

In [ ]:
# Rasio pengelolaan total = (B + C) / A
df_fe['rasio_pengelolaan'] = ((df_fe[B] + df_fe[C]) / df_fe[A]).clip(0, 1)

# Rasio pengurangan = B / A  (3R, komposting, bank sampah, dll.)
df_fe['rasio_pengurangan'] = (df_fe[B] / df_fe[A]).clip(0, 1)

# Rasio penanganan = C / A  (TPA, pengolahan akhir)
df_fe['rasio_penanganan']  = (df_fe[C] / df_fe[A]).clip(0, 1)

# Recycling rate = (D + E) / A
df_fe['recycling_rate']    = ((df_fe[D] + df_fe[E]) / df_fe[A]).clip(0, 1)

print('Statistik fitur efisiensi pengelolaan:')
print(df_fe[['rasio_pengelolaan', 'rasio_pengurangan',
             'rasio_penanganan', 'recycling_rate']].describe().round(3))

### 5.2 Fitur Efisiensi Fasilitas TPA

In [ ]:
# Utilisasi TPA = sampah masuk TPA / timbulan daerah
df_fe['utilisasi_tpa'] = (
    df_fe['Sampahmasuk (ton/thn)'] / df_fe[A]
).clip(0, 1)

# Efisiensi landfill = sampah masuk landfill / total sampah masuk TPA
# Nilai mendekati 1 berarti hampir semua sampah langsung dilandfill (kurang diolah dulu)
df_fe['efisiensi_landfill'] = (
    df_fe['Sampahmasuk Landfill (ton/thn)'] / df_fe['Sampahmasuk (ton/thn)']
).clip(0, 1)

# Rasio recovery pemulung = RecoveryPemulung / sampah masuk TPA
df_fe['rasio_recovery_pemulung'] = (
    df_fe['RecoveryPemulung (ton/thn)'] / df_fe['Sampahmasuk (ton/thn)']
).clip(0, 1)

# Rasio sampah organik terolah per ton sampah masuk
df_fe['rasio_organik_terolah'] = (
    df_fe['Sampah Organikterolah (ton/thn)'] / df_fe['Sampahmasuk (ton/thn)']
).clip(0, 1)

print('Statistik fitur efisiensi TPA:')
print(df_fe[['utilisasi_tpa', 'efisiensi_landfill',
             'rasio_recovery_pemulung', 'rasio_organik_terolah']].describe().round(3))

### 5.3 Fitur Usia dan Kapasitas TPA

In [ ]:
TAHUN_REFERENSI = 2025

# Usia operasi TPA = tahun referensi - tahun awal operasi
df_fe['usia_operasi_tpa'] = (TAHUN_REFERENSI - df_fe['Tahun Awal Operasi']).clip(lower=0)

# Sampah per hektar = sampah masuk / luas TPA  (ton/ha/tahun)
# inf muncul jika Luas (hektar) = 0 → ganti dengan NaN
df_fe['sampah_per_hektar'] = (
    df_fe['Sampahmasuk (ton/thn)'] / df_fe['Luas (hektar)']
).replace([np.inf, -np.inf], np.nan)

# Proporsi landfill aktif terhadap total lahan TPA
df_fe['proporsi_landfill_aktif'] = (
    df_fe['Luas Landfill Aktif (m2)'] / (df_fe['Luas (hektar)'] * 10_000)
).clip(0, 1)

print('Statistik fitur usia dan kapasitas TPA:')
print(df_fe[['usia_operasi_tpa', 'sampah_per_hektar',
             'proporsi_landfill_aktif']].describe().round(2))

### 5.4 Encoding Fitur Ordinal (Infrastruktur TPA)

In [ ]:
# Frekuensi penutupan zona aktif → skor 0-7 (makin sering makin baik)
MAP_PENUTUPAN = {
    'Setiap Hari':        7,
    '2-3x Seminggu':      4,
    'Seminggu Sekali':    3,
    'Dua Minggu Sekali':  2,
    'Sebulan Sekali':     1,
    '3-4 Bulan Sekali':   1,
    'Setahun Sekali':     0,
    'Tidak Ditutup':      0,
}

# Ketersediaan jembatan timbang → skor 0-2
MAP_JEMBATAN = {
    'Ada & Berfungsi':  2,
    'Ada tapi Rusak':   1,
    'Tidak Ada':        0,
}

# Ada drainase → biner
MAP_DRAINASE = {'ADA': 1, 'TIDAK ADA': 0}

df_fe['skor_penutupan']   = df_fe['Penutupan Sampah Zona Aktif'].map(MAP_PENUTUPAN)
df_fe['skor_jembatan']    = df_fe['Jembatan Timbang'].map(MAP_JEMBATAN)
df_fe['skor_drainase']    = df_fe['Ada Drainase'].map(MAP_DRAINASE)

# Skor infrastruktur gabungan (range 0-9); NaN diisi 0 hanya untuk agregasi ini
df_fe['skor_infrastruktur'] = (
    df_fe['skor_penutupan'].fillna(0)
    + df_fe['skor_jembatan'].fillna(0)
    + df_fe['skor_drainase'].fillna(0)
)

print('Distribusi skor infrastruktur TPA (0 = buruk, 9 = sangat baik):')
print(df_fe['skor_infrastruktur'].value_counts().sort_index().to_string())
print()
print(df_fe[['skor_penutupan', 'skor_jembatan', 'skor_drainase',
             'skor_infrastruktur']].describe().round(2))

### 5.5 Encoding Fitur Nominal (One-Hot)

In [ ]:
OHE_COLS = ['Jenis', 'Pengelola', 'Sistem Operasional']

df_fe = pd.get_dummies(df_fe, columns=OHE_COLS, prefix=OHE_COLS, dummy_na=True)

# Cast bool hasil OHE ke int8 agar hemat memori dan kompatibel dengan model
bool_cols = df_fe.select_dtypes(include='bool').columns
df_fe[bool_cols] = df_fe[bool_cols].astype(np.int8)

new_ohe = [c for c in df_fe.columns if any(c.startswith(p + '_') for p in OHE_COLS)]
print(f'Kolom OHE yang dihasilkan ({len(new_ohe)}):')
print(new_ohe)

### 5.6 Drop Kolom Raw yang Sudah Ter-replace

In [ ]:
# Kolom yang sudah digantikan oleh fitur engineered di atas
COLS_ENCODED_REPLACED = [
    'Penutupan Sampah Zona Aktif',  # -> skor_penutupan
    'Jembatan Timbang',             # -> skor_jembatan
    'Ada Drainase',                 # -> skor_drainase
]

COLS_RAW_SIPSN = [A, B, C, D, E]   # -> rasio_*

COLS_RAW_TPA = [
    'Sampahmasuk (ton/thn)',
    'Sampahmasuk Landfill (ton/thn)',
    'Sampah Organikterolah (ton/thn)',
    'Sampah An-Organikterolah (ton/thn)',
    'RecoveryPemulung (ton/thn)',
]                                   # -> utilisasi_*, efisiensi_*, rasio_*

COLS_REPLACED = [
    'Tahun Awal Operasi',           # -> usia_operasi_tpa
    'Luas Landfill Aktif (m2)',     # -> proporsi_landfill_aktif
]

ALL_REPLACED = (
    COLS_ENCODED_REPLACED + COLS_RAW_SIPSN + COLS_RAW_TPA + COLS_REPLACED
)
ALL_REPLACED = [c for c in ALL_REPLACED if c in df_fe.columns]

df_fe = df_fe.drop(columns=ALL_REPLACED)

print(f'{len(ALL_REPLACED)} kolom raw di-drop.')
print(f'Shape setelah drop: {df_fe.shape}')

## 6. Dataset Final

Kolom diurutkan secara logis: **identifikasi wilayah → fitur SIPSN → fitur TPA →
infrastruktur → one-hot encoding → flag ketersediaan data**.

In [ ]:
ID_COLS    = ['Provinsi', 'Kabupaten/Kota']
FEAT_SIPSN = ['rasio_pengelolaan', 'rasio_pengurangan',
              'rasio_penanganan', 'recycling_rate']
FEAT_TPA   = ['Luas (hektar)', 'Jml Sumur Pantau',
              'utilisasi_tpa', 'efisiensi_landfill',
              'rasio_recovery_pemulung', 'rasio_organik_terolah',
              'sampah_per_hektar', 'proporsi_landfill_aktif',
              'usia_operasi_tpa']
FEAT_INFRA = ['skor_penutupan', 'skor_jembatan',
              'skor_drainase', 'skor_infrastruktur']
OHE_FINAL  = [c for c in df_fe.columns
              if any(c.startswith(p + '_') for p in ['Jenis', 'Pengelola', 'Sistem Operasional'])]
FLAG_COLS  = ['Ada Data SIPSN', 'Ada Data TPA']

ordered_cols = ID_COLS + FEAT_SIPSN + FEAT_TPA + FEAT_INFRA + OHE_FINAL + FLAG_COLS
ordered_cols = [c for c in ordered_cols if c in df_fe.columns]
remaining    = [c for c in df_fe.columns if c not in ordered_cols]
df_final     = df_fe[ordered_cols + remaining]

print(f'Shape dataset final: {df_final.shape}')
print(f'\nDaftar kolom ({len(df_final.columns)}):')
for i, col in enumerate(df_final.columns, 1):
    print(f'  {i:>2}. {col}')

### 6.1 Missing Values Dataset Final

In [ ]:
missing_final = (df_final.isnull().sum() / len(df_final) * 100).round(1)
missing_final = missing_final[missing_final > 0].sort_values(ascending=False)

print(f'Total baris  : {len(df_final)}')
print(f'Total kolom  : {len(df_final.columns)}')
print(f'Kolom lengkap: {(df_final.isnull().sum() == 0).sum()} dari {len(df_final.columns)}')
print()
if missing_final.empty:
    print('Tidak ada missing values.')
else:
    print('Kolom dengan missing values:')
    print(missing_final.to_string())

### 6.2 Statistik Deskriptif Dataset Final

In [ ]:
df_final.describe().round(3)

### 6.3 Distribusi Fitur Engineered Utama

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

plot_cols = [
    'rasio_pengelolaan', 'rasio_pengurangan', 'rasio_penanganan', 'recycling_rate',
    'utilisasi_tpa', 'skor_infrastruktur', 'usia_operasi_tpa', 'sampah_per_hektar',
]

for ax, col in zip(axes, plot_cols):
    data = df_final[col].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xlabel('Nilai', fontsize=8)
    ax.set_ylabel('Frekuensi', fontsize=8)
    ax.tick_params(labelsize=7)

fig.suptitle('Distribusi Fitur Engineered Utama', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.4 Simpan Dataset Final

In [ ]:
df_final.to_csv('Dataset/final_waste_management.csv', index=False)

print(f'Dataset final disimpan: Dataset/final_waste_management.csv')
print(f'Shape                 : {df_final.shape}')
print()
print(df_final.head())